# MCD-rPPG Dataset Preprocessing Notebook

**Database-Driven Preprocessing with Testing & Verification**

This notebook processes the MCD-rPPG dataset using the database CSV file (db.csv).
It includes a **TEST PHASE** that validates preprocessing on 2 random videos before full processing.

## Dataset Structure

### Database Columns:
- patient_id, weight, height, bmi, age, sex
- upper_ap, lower_ap (blood pressure)
- saturation, temperature, pulse, stress
- step: 'before' or 'after'
- camera: 'IriunWebcam', 'FullHDwebcam', 'USBVideo'
- view: 'front', 'left', 'right'
- video, ppg, meta, ppg_sync, ecg: relative paths

### File Formats:
- Video: .avi, ~30 FPS
- PPG: .PW, format: value timestamp
- Meta: .txt, format: frame_number timestamp
- PPG Sync: .txt, format: frame_number sync_value

## Workflow:
1. Setup & Configuration
2. Load Database
3. Initialize MediaPipe
4. TEST PHASE: Process 2 random videos with visualization
5. Preprocessing Functions
6. Process Full Dataset
7. Save Preprocessed Data
8. Verify Output

---

## 1. Setup and Configuration

In [ ]:
!pip install mediapipe>=0.10.11 opencv-python-headless tqdm scikit-image numpy pandas scipy matplotlib

In [ ]:
import os, sys, numpy as np, pandas as pd, cv2
from tqdm import tqdm
from datetime import datetime
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
import random
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
print('All imports successful!')
print(f'MediaPipe version: {mp.__version__}')

In [ ]:
class Config:
    db_path = None
    dataset_root = None
    output_path = './preprocessed_data'
    window_size = 256
    stride = 64
    target_size = (128, 128)
    padding = 20
    min_face_size = 64
    ppg_low_freq = 0.75
    ppg_high_freq = 4.0
    frame_rate = 30.0
    ppg_rate = 100.0
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    random_state = 42
config = Config()
print('Configuration initialized')

In [ ]:
config.db_path = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
config.dataset_root = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88'
if os.path.exists(config.db_path):
    print(f'Database CSV: {config.db_path}')
if os.path.exists(config.dataset_root):
    print(f'Dataset root: {config.dataset_root}')

---

## 2. Load Database

In [ ]:
db = pd.read_csv(config.db_path)
print(f'Loaded {len(db)} rows')
print(f'Columns: {list(db.columns)}')
display(db.head(2))

---

## 3. Initialize MediaPipe

In [ ]:
def initialize_mediapipe():
    base_options = python.BaseOptions(model_asset_path=None)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(options)
detector = initialize_mediapipe()
print('MediaPipe initialized')

---

## 4. File Parsing Functions

In [ ]:
def parse_pw_file(pw_path):
    ppg_values, timestamps = [], []
    with open(pw_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    ppg_values.append(float(parts[0]))
                    timestamps.append(datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f'))
                except: pass
    return np.array(ppg_values), np.array(timestamps)

def parse_meta_file(meta_path):
    meta_data = {}
    with open(meta_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    meta_data[int(parts[0])] = datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f')
                except: pass
    return meta_data

def parse_ppg_sync_file(sync_path):
    sync_data = {}
    with open(sync_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    sync_data[int(parts[0])] = float(parts[1])
                except: pass
    return sync_data

---

## 5. PPG-Video Alignment Function

This is the CORE function that aligns PPG signals with video frames using timestamp interpolation.

In [ ]:
def align_ppg_with_video(ppg_values, ppg_timestamps, meta_data, frame_rate=30.0):
    if len(meta_data) == 0 or len(ppg_timestamps) == 0:
        return np.zeros(len(meta_data))
    first_ppg = ppg_timestamps[0]
    first_meta = list(meta_data.values())[0]
    ppg_seconds = [(ts - first_ppg).total_seconds() for ts in ppg_timestamps]
    sorted_frames = sorted(meta_data.keys())
    meta_seconds = [(meta_data[f] - first_meta).total_seconds() for f in sorted_frames]
    interp_func = interp1d(ppg_seconds, ppg_values, kind='linear', fill_value='extrapolate')
    aligned_ppg = interp_func(meta_seconds)
    return aligned_ppg

def preprocess_ppg(ppg, low_freq=0.75, high_freq=4.0, fs=100.0):
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, ppg)
    return (filtered - filtered.mean()) / filtered.std()

---

## 6. Video Processing Functions

In [ ]:
def load_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): raise ValueError(f'Could not open: {video_path}')
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.array(frames)

def detect_face(frame, detector, frame_idx):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = detector.detect_for_video(mp_image, frame_idx)
    if result and result.face_landmarks:
        lm = result.face_landmarks[0]
        h, w = frame.shape[:2]
        return np.array([(p.x * w, p.y * h) for p in lm], dtype=np.float32)
    return None

def process_frame(frame, detector, frame_idx, prev_landmarks=None, min_size=64):
    landmarks = detect_face(frame, detector, frame_idx)
    if landmarks is None: return None, prev_landmarks if prev_landmarks is not None else None
    x_min, x_max = landmarks[:, 0].min(), landmarks[:, 0].max()
    y_min, y_max = landmarks[:, 1].min(), landmarks[:, 1].max()
    w, h = x_max - x_min, y_max - y_min
    if w < min_size or h < min_size: return None, landmarks
    pad = 20
    x_min = max(0, int(x_min) - pad)
    x_max = min(frame.shape[1], int(x_max) + pad)
    y_min = max(0, int(y_min) - pad)
    y_max = min(frame.shape[0], int(y_max) + pad)
    face = frame[y_min:y_max, x_min:x_max]
    return cv2.resize(face, (128, 128)), landmarks

In [ ]:
def extract_chunks(video, ppg, window_size, stride, video_fps=30.0, ppg_fps=100.0):
    chunks, ppg_chunks = [], []
    num_frames = video.shape[0]
    ppg_per_frame = ppg_fps / video_fps
    for start in range(0, num_frames - window_size + 1, stride):
        end = start + window_size
        video_chunk = video[start:end]
        ppg_start = int(start * ppg_per_frame)
        ppg_end = int(end * ppg_per_frame)
        ppg_chunk = ppg[ppg_start:ppg_end]
        if len(ppg_chunk) == window_size:
            chunks.append(video_chunk)
            ppg_chunks.append(ppg_chunk)
    return np.array(chunks), np.array(ppg_chunks)